# Notebook 02: Hệ thống câu hỏi nghiên cứu và Tổng quan phân tích chuyên sâu (Master Hub)
**Đề tài:** Phân tích sự cố bất lợi thuốc thú y (OpenFDA Animal & Veterinary Adverse Events) giai đoạn 2025 và Đối chiếu xu hướng 2026 (Q1)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Nhóm thực hiện:** Global Ban

> **Phân rã notebook 02 theo câu hỏi:**  
> Để tối ưu hóa quy trình làm việc nhóm, chống xung đột phiên bản Git và đảm bảo tính độc lập chuyên sâu, toàn bộ phân tích chi tiết cho 5 câu hỏi Non-ML được tổ chức thành **5 Notebook riêng biệt**:
> 1. **[`02_q1_analysis.ipynb`](02_q1_analysis.ipynb)**: **Câu hỏi Q1** — Phản ứng VeDDRA & Xu hướng theo quý
> 2. **[`02_q2_analysis.ipynb`](02_q2_analysis.ipynb)**: **Câu hỏi Q2** — Đặc điểm sinh học & Biến cố nghiêm trọng
> 3. **[`02_q3_analysis.ipynb`](02_q3_analysis.ipynb)**: **Câu hỏi Q3** — Tỷ suất đồng xuất hiện ROR Hoạt chất x Triệu chứng
> 4. **[`02_q4_analysis.ipynb`](02_q4_analysis.ipynb)**: **Câu hỏi Q4** — Dùng thuốc sai nhãn Off-label & Kết cục y khoa
> 5. **[`02_q5_analysis.ipynb`](02_q5_analysis.ipynb)**: **Câu hỏi Q5** — Dịch tễ đàn & Tỷ lệ ảnh hưởng ở đàn
>
> File Master này (`02_analysis.ipynb`) đóng vai trò là **Trung tâm Điều phối**: Trình bày Mục 3.3 (Hệ thống câu hỏi và Ma trận độ sâu D1–D6), cung cấp liên kết trực tiếp sang từng câu hỏi Q1–Q5 và tổng hợp Kết quả then chốt của 5 câu hỏi phân tích.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_processed
from src.visualization import set_project_plot_style
set_project_plot_style()

# Tải 4 bảng dữ liệu đã qua tiền xử lý chuẩn hóa từ data/processed/
data = load_processed(PROJECT_ROOT / 'data' / 'processed')
reports = data['reports']
drugs = data['drugs']
reactions = data['reactions']
outcomes = data['outcomes']
print(f"[Data Loaded] Reports: {reports.shape}, Drugs: {drugs.shape}, Reactions: {reactions.shape}, Outcomes: {outcomes.shape}")

---
## 1. Tổng quan hệ thống câu hỏi nghiên cứu và độ sâu


| Mã | Chủ sở hữu | Câu hỏi nghiên cứu | Tiêu chí độ sâu đáp ứng | Notebook thực thi |
|---|---|---|---|---|
| **Q1** | **M1 — Khang** | Những phản ứng bất lợi (VeDDRA terms) nào chiếm tỷ trọng cao nhất ở Chó, Mèo và các loài khác trong năm 2025, và xu hướng biến động theo quý (kèm dấu hiệu mùa vụ trong phạm vi 1 năm) của chúng diễn biến ra sao khi bước sang nửa đầu năm 2026 (Q1)? | **D1** (Combining tables), **D2** (Reshaping & time series), **D5** (Quantifying uncertainty) | [`02_q1_analysis.ipynb`](02_q1_analysis.ipynb) |
| **Q2** | **M2 — Nhi** | Những đặc điểm sinh học nào (giai đoạn sống, trọng lượng cơ thể, giới tính) có liên quan đến các biến cố bất lợi nghiêm trọng, sau khi đã kiểm soát yếu tố loài và số lượng thuốc sử dụng? | **D3** (Derived variables), **D4** (Controlling for confounder), **D5** (CI/Hypothesis test), **D6** (Robustness check) | [`02_q2_analysis.ipynb`](02_q2_analysis.ipynb) |
| **Q3** | **M3 — Kiệt** | Các hoạt chất (`active_ingredients`) và đường dùng (`route`) nào có tỷ suất đồng xuất hiện (Reporting Odds Ratio) đặc trưng nhất với từng nhóm phản ứng cụ thể, và cấu trúc này khác biệt ra sao giữa **Safety Issue** và **Lack Of Expected Effectiveness** trong năm 2025? | **D1** (Multi-table merge), **D2** (Pivot/melt matrix), **D4** (Subgroup control), **D5** (Odds Ratio & 95% CI) | [`02_q3_analysis.ipynb`](02_q3_analysis.ipynb) |
| **Q4** | **M3 — Kiệt** | Việc dùng thuốc sai nhãn (`off-label`) kết hợp với đa trị liệu (`polypharmacy`) trong năm 2025 có liên hệ như thế nào với mức độ nghiêm trọng (`serious_ae`) và kết cục (`outcome`) sau khi kiểm soát biến nhiễu là **loài** và **đối tượng báo cáo (`primary_reporter`)**? | **D1** (Combining tables), **D3** (Derived indicators), **D4** (Controlling for confounder), **D6** (Robustness check) | [`02_q4_analysis.ipynb`](02_q4_analysis.ipynb) |
| **Q5** | **M2 — Nhi** | Những yếu tố nào có liên quan đến tỷ lệ động vật bị ảnh hưởng trong tổng số động vật được điều trị ở mỗi đàn? | **D1** (Multi-table join for herd data), **D3** (Derived attack rate), **D4** (Subgroup / species control), **D5** (CI & regression), **D6** (Robustness: exclude attack_rate > 1 vs. keep) | [`02_q5_analysis.ipynb`](02_q5_analysis.ipynb) |
| **Q6 (ML)** | **M1 — Khang** | Huấn luyện mô hình trên dữ liệu 2025 (Q1–Q4) để dự đoán khả năng một báo cáo trong năm 2026 (Q1) là sự cố nghiêm trọng (`serious_ae`) dựa trên thông tin ban đầu lúc tiếp nhận. | **ML Checklist 1–6** (Time-based split, Pipeline, Leakage-free, Tuning, Error Analysis, Permutation Importance) | [`03_ml_question.ipynb`](03_ml_question.ipynb) |


---
## 2. Câu hỏi Q1: Phản ứng VeDDRA hàng đầu & Xu hướng Biến động theo Quý
- **Chủ sở hữu:** Đinh Công Khang
- **Notebook chi tiết:** [`02_q1_analysis.ipynb`](02_q1_analysis.ipynb)
- **Tóm tắt phát hiện:**
  - *Top phản ứng VeDDRA theo loài:* (Sẽ được trích xuất từ `02_q1_analysis.ipynb` khi hoàn tất phân tích)
  - *Dấu hiệu mùa vụ & Biến động 2025 $\rightarrow$ 2026 Q1:* (Sẽ được trích xuất từ `02_q1_analysis.ipynb`)
  - *Giá trị p-value kiểm định $\chi^2$ & Bootstrap 95% CI:* (Sẽ được trích xuất từ `02_q1_analysis.ipynb`)


---
## 3. Câu hỏi Q2: Đặc điểm Sinh học & Nguy cơ Biến cố Nghiêm trọng
- **Chủ sở hữu:** Trần Ngô Uyên Nhi
- **Notebook chi tiết:** [`02_q2_analysis.ipynb`](02_q2_analysis.ipynb)
- **Tóm tắt phát hiện:**
  - *Ảnh hưởng của giai đoạn sống & Cân nặng:* Trên `71,607` ca hoàn chỉnh (`58,634` Chó, `12,973` Mèo), thú già (`Senior`) tăng nguy cơ ở cả hai loài, còn cân nặng có tác động **đảo chiều**: Chó nặng cân có nguy cơ cao hơn (`Q1` 42.42% → `Q4` 47.68%), ngược lại Mèo nhẹ cân có nguy cơ cao hơn (`Q1` 38.46% → `Q4` 31.71%). Đặc biệt, đường cong chữ U ở Mèo non (`Junior`) biến mất khi kiểm soát cân nặng và đa trị liệu.
  - *Tỷ suất chênh điều chỉnh (Adjusted Odds Ratio) & 95% CI:* Ở Chó: `Senior` `aOR = 1.293 (1.248–1.339)`, `Junior` `aOR = 0.762 (0.712–0.816)`, cân nặng `+1 SD` `aOR = 1.065 (1.047–1.083)`, `3+ thuốc` `aOR = 1.749 (1.678–1.823)`. Ở Mèo: `Senior` `aOR = 1.298 (1.201–1.404)`, `Junior` `aOR = 1.050 (0.903–1.223, p = 0.525)`, cân nặng `+1 SD` `aOR = 0.922 (0.883–0.962)`, `3+ thuốc` `aOR = 2.827 (2.586–3.089)`. Giới tính (`Female vs Male`) không có ý nghĩa lâm sàng.
  - *Kết quả kiểm tra độ vững D6:* Toàn bộ chiều hướng và ý nghĩa thống kê giữ nguyên vững vàng qua 4 kịch bản (`R1` cắt 1%–99%, `R2` giữ nhóm `'Unknown'`, `R3` kiểm soát thêm `primary_reporter`, `R4` chỉ xét năm 2025).


---
## 4. Câu hỏi Q3: Tỷ suất Đồng xuất hiện (ROR) Hoạt chất x Triệu chứng (Safety vs Efficacy)
- **Chủ sở hữu:** Huỳnh Tuấn Kiệt
- **Notebook chi tiết:** [`02_q3_analysis.ipynb`](02_q3_analysis.ipynb)
- **Tóm tắt phát hiện:**
  - *Cặp Hoạt chất — Triệu chứng có tín hiệu an toàn mạnh nhất (ROR > 2):* (Sẽ được trích xuất từ `02_q3_analysis.ipynb` khi hoàn tất phân tích)
  - *Khác biệt cấu trúc giữa Safety Issue vs Lack of Efficacy:* (Sẽ được trích xuất từ `02_q3_analysis.ipynb`)
  - *Khoảng tin cậy 95% CI và kết quả phân tầng loài:* (Sẽ được trích xuất từ `02_q3_analysis.ipynb`)


---
## 5. Câu hỏi Q4: Sử dụng Thuốc Sai Nhãn (Off-label), Đa Trị Liệu & Kết cục Y khoa
- **Chủ sở hữu:** Huỳnh Tuấn Kiệt
- **Notebook chi tiết:** [`02_q4_analysis.ipynb`](02_q4_analysis.ipynb)
- **Tóm tắt phát hiện:**
  - *Tỷ lệ gia tăng nguy cơ tử vong khi dùng off-label:* (Sẽ được trích xuất từ `02_q4_analysis.ipynb` khi hoàn tất phân tích)
  - *Tác động cộng hưởng giữa Off-label và Đa trị liệu (Polypharmacy):* (Sẽ được trích xuất từ `02_q4_analysis.ipynb`)
  - *Kiểm soát biến nhiễu người báo cáo (`primary_reporter`) và kết quả D6:* (Sẽ được trích xuất từ `02_q4_analysis.ipynb`)


---
## 6. Câu hỏi Q5: Các yếu tố liên quan đến Tỷ lệ Động vật Bị ảnh hưởng ở Đàn (Herd Attack Rate)
- **Chủ sở hữu:** Trần Ngô Uyên Nhi
- **Notebook chi tiết:** [`02_q5_analysis.ipynb`](02_q5_analysis.ipynb)
- **Tóm tắt phát hiện:**
  - *Mức độ ảnh hưởng trung vị theo loài đàn và đường dùng thuốc:* (Sẽ được trích xuất từ `02_q5_analysis.ipynb` khi hoàn tất phân tích)
  - *Giá trị kiểm định Kruskal-Wallis & Bootstrap 95% CI:* (Sẽ được trích xuất từ `02_q5_analysis.ipynb`)
  - *Kiểm tra độ vững D6 khi xử lý các ca attack_rate > 1:* (Sẽ được trích xuất từ `02_q5_analysis.ipynb`)
